# Static Image vs. FEA Significance Tests

This notebook implements the statistical comparison between the static image-based and FEA-based FER models.

The test set contains **378 reenactments**. Each reenactment contributes:

- one central-view image prediction
- one side-view image prediction
- one FEA prediction

The two image views belonging to the same reenactment are therefore not treated as independent experimental units.

## Analysis plan

1. Primary comparison: compare pooled image accuracy with FEA accuracy using a paired cluster bootstrap over reenactments.
2. Sensitivity check: test the same reenactment-level mean difference with a one-sample t-test.
3. Secondary analysis: run exact McNemar tests for Central vs. FEA, Side vs. FEA, and Central vs. Side, followed by Holm correction.
4. Participant-level heterogeneity check: summarize the pooled Image-vs.-FEA difference separately for each of the eight participants.

### Note on the primary resampling test

A simple paired label-permutation/sign-flip test is not used here because the reenactment-level image score has support $\{0, 0.5, 1\}$, whereas FEA correctness has support $\{0, 1\}$. Exchanging the two measurements within a pair would therefore require an exchangeability assumption that does not hold by construction.

Instead, the primary significance test uses a **centered paired cluster bootstrap** on the reenactment-level accuracy differences. The same clustering scheme is used to obtain the 95% confidence interval for the accuracy difference.


## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests


PREDICTIONS_PATH = Path('static_test_predictions.csv')

RANDOM_SEED = 42

# Use many resamples for stable final confidence intervals and p-values.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05

## 2. Load and Validate Predictions

The prediction CSV is expected to contain one row per image sample and the following columns:

- `sample_id`
- `reenactment_id`
- `camera_index`
- `true_label_id`
- `image_pred_id`
- `fea_pred_id`

Camera index `0` denotes the central view and camera index `1` the side view.

In [2]:
prediction_df = pd.read_csv(PREDICTIONS_PATH)

print(f"Rows: {len(prediction_df)}")
print(f"Columns: {len(prediction_df.columns)}")
prediction_df.head()


Rows: 756
Columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.004558,0,Anger,0.677313,0.030212,0.002842,0.003746,0.026476,0.252719,0.006691
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.004558,4,Neutral,0.095327,0.056766,0.039777,0.011597,0.752703,0.013406,0.030424
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000017,5,Sadness,0.000005,0.000117,0.000001,0.000006,0.000001,0.999867,0.000003
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000017,5,Sadness,0.000149,0.000589,0.000038,0.000047,0.000028,0.999063,0.000088
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000797,3,Happiness,0.000376,0.001057,0.000616,0.995559,0.000666,0.000936,0.000790


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "image_pred_id",
    "fea_pred_id",
}

missing_columns = required_columns - set(prediction_df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378
assert set(prediction_df["camera_index"].unique()) == {0, 1}

samples_per_reenactment = prediction_df.groupby("reenactment_id").size()
assert samples_per_reenactment.eq(2).all()

views_per_reenactment = prediction_df.groupby("reenactment_id")["camera_index"].nunique()
assert views_per_reenactment.eq(2).all()

true_labels_per_reenactment = prediction_df.groupby("reenactment_id")["true_label_id"].nunique()
assert true_labels_per_reenactment.eq(1).all()

participants_per_reenactment = prediction_df.groupby("reenactment_id")["participant_id"].nunique()
assert participants_per_reenactment.eq(1).all()
assert prediction_df["participant_id"].nunique() == 8

fea_predictions_per_reenactment = prediction_df.groupby("reenactment_id")["fea_pred_id"].nunique()
assert fea_predictions_per_reenactment.eq(1).all()

print("Prediction-table structure validated.")


Prediction-table structure validated.


## 3. Construct Reenactment-Level Analysis Table

For each reenactment, $C_i$ indicates whether the central-view image was classified correctly, $S_i$ indicates whether the side-view image was classified correctly, and $F_i$ indicates whether the FEA sample was classified correctly.

The pooled image contribution of reenactment $i$ is $I_i = \frac{C_i + S_i}{2}$.

Averaging $I_i$ across all 378 reenactments exactly reproduces the pooled image accuracy over all 756 images.


In [4]:
prediction_df = prediction_df.copy()

prediction_df["image_correct"] = (prediction_df["image_pred_id"] == prediction_df["true_label_id"])

prediction_df["fea_correct"] = (prediction_df["fea_pred_id"] == prediction_df["true_label_id"])


In [5]:
image_correct_by_view = (
    prediction_df
    .pivot(
        index="reenactment_id",
        columns="camera_index",
        values="image_correct",
    )
    .rename(columns={
        0: "central_correct",
        1: "side_correct",
    })
)

fea_correct = prediction_df.groupby("reenactment_id")["fea_correct"].first()
participant_id = prediction_df.groupby("reenactment_id")["participant_id"].first()

analysis_df = image_correct_by_view.join(fea_correct).join(participant_id)

analysis_df["central_correct"] = analysis_df["central_correct"].astype(bool)
analysis_df["side_correct"] = analysis_df["side_correct"].astype(bool)
analysis_df["fea_correct"] = analysis_df["fea_correct"].astype(bool)

analysis_df["image_correct_mean"] = (
    analysis_df["central_correct"].astype(float)
    + analysis_df["side_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()


,central_correct,side_correct,fea_correct,participant_id,image_correct_mean
reenactment_id,,,,,
1700478995850-2-1-1-0-0,True,True,False,1,1.0
1700478998549-2-1-1-1-5,True,True,True,1,1.0
1700479001137-2-1-1-2-3,True,True,True,1,1.0
1700479004312-2-1-1-3-0,False,False,True,1,0.0
1700479005401-2-1-1-4-0,True,True,True,1,1.0


## 4. Verify Reported Performance

In [6]:
pooled_image_accuracy = analysis_df["image_correct_mean"].mean()
fea_accuracy = analysis_df["fea_correct"].mean()

central_accuracy = analysis_df["central_correct"].mean()
side_accuracy = analysis_df["side_correct"].mean()

accuracy_difference = fea_accuracy - pooled_image_accuracy

print(f"Pooled image accuracy: {pooled_image_accuracy:.4%}")
print(f"FEA accuracy:          {fea_accuracy:.4%}")
print(f"Central accuracy:      {central_accuracy:.4%}")
print(f"Side accuracy:         {side_accuracy:.4%}")
print(f"FEA - Image:           {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(pooled_image_accuracy, prediction_df["image_correct"].mean())

assert np.isclose(fea_accuracy,prediction_df["fea_correct"].mean())

assert prediction_df["image_correct"].sum() == 528
assert np.isclose(pooled_image_accuracy, 528 / 756)

# The same FEA prediction occurs for both views, so the 756-row table
# contains exactly twice the number of correct reenactment-level FEA predictions.
assert prediction_df["fea_correct"].sum() == 542
assert analysis_df["fea_correct"].sum() == 271
assert np.isclose(fea_accuracy, 271 / 378)


Pooled image accuracy: 69.8413%
FEA accuracy:          71.6931%
Central accuracy:      72.7513%
Side accuracy:         66.9312%
FEA - Image:           1.85 percentage points


## 5. Primary Image vs. FEA Comparison

The primary estimand is the difference between the reported overall accuracies, $\Delta = \mathrm{Accuracy}_{FEA} - \mathrm{Accuracy}_{Image}$.

For each reenactment, $d_i = F_i - I_i$. The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor FEA.


In [7]:
def paired_cluster_bootstrap(differences: np.ndarray,
                             n_bootstrap: int = N_BOOTSTRAP,
                             batch_size: int = BOOTSTRAP_BATCH_SIZE,
                             seed: int = RANDOM_SEED,
                             alpha: float = ALPHA) -> dict:
    
    differences = np.asarray(differences, dtype=float)

    if differences.ndim != 1 or differences.size == 0:
        raise ValueError("differences must be a nonempty one-dimensional array.")
    if not np.isin(differences, [-1.0, -0.5, 0.0, 0.5, 1.0]).all():
        raise ValueError("Expected paired accuracy differences in {-1, -0.5, 0, 0.5, 1}.")

    n = len(differences)
    observed_difference = differences.mean()

    doubled_differences = (2 * differences).astype(np.int64)
    observed_sum = doubled_differences.sum()

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=float)
    n_extreme = 0

    for start in range(0, n_bootstrap, batch_size):
        end = min(start + batch_size, n_bootstrap)
        current_batch_size = end - start

        # Ordinary paired bootstrap for the confidence interval.
        bootstrap_indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:end] = differences[bootstrap_indices].mean(axis=1)

        # Centered-bootstrap null test evaluated in exact integer arithmetic:
        # |mean(d*) - mean(d)| >= |mean(d)| is equivalent to |S* - S| >= |S|,
        # where q_i = 2 d_i, S = sum(q_i), and S* = sum(q_i*).
        null_indices = rng.integers(0, n, size=(current_batch_size, n))
        null_sums = doubled_differences[null_indices].sum(axis=1)

        n_extreme += np.count_nonzero(
            np.abs(null_sums - observed_sum) >= abs(observed_sum)
        )

    ci_low, ci_high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    p_value = (n_extreme + 1) / (n_bootstrap + 1)

    return {
        "n": n,
        "difference": observed_difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_bootstrap": n_bootstrap
    }


In [8]:
reenactment_differences = (
    analysis_df["fea_correct"] - analysis_df["image_correct_mean"].astype(float)
).to_numpy()

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "FEA - pooled image",
    "n_reenactments": primary_result["n"],
    "image_accuracy": pooled_image_accuracy,
    "fea_accuracy": fea_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"],
}])

primary_result_df


,comparison,n_reenactments,image_accuracy,fea_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,FEA - pooled image,378,0.698413,0.716931,1.851852,-3.306878,6.878307,0.493626,1000000


In [9]:
row = primary_result_df.iloc[0]

print(
    f"FEA - pooled image accuracy difference: "
    f"{row['difference_pp']:.2f} percentage points"
)
print(
    f"{100 * (1 - ALPHA):.0f}% bootstrap CI: "
    f"[{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points"
)
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")


FEA - pooled image accuracy difference: 1.85 percentage points
95% bootstrap CI: [-3.31, 6.88] percentage points
Two-sided bootstrap p-value: 0.493625506374


## 6. Sensitivity Check

As a sensitivity analysis, the same 378 reenactment-level differences $d_i = F_i - I_i$ are tested with a one-sample t-test against a mean of zero.

This is not a separate primary hypothesis test. It checks whether a conventional parametric test leads to the same substantive conclusion as the bootstrap analysis. No multiplicity correction is applied to this sensitivity check.


In [10]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "FEA - pooled image",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df


,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,FEA - pooled image,378,1.851852,0.70845,377,0.479103


In [11]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")


Mean difference: 1.85 percentage points
t(377) = 0.708
Two-sided sensitivity-check p-value: 0.479103456351


## 7. Secondary View-Specific Comparisons

The secondary analysis uses three exact paired McNemar tests over the same 378 reenactments:

1. Central vs. FEA
2. Side vs. FEA
3. Central vs. Side

For two paired binary correctness variables $A$ and $B$, only discordant pairs contribute to McNemar's test. Because the three pairwise questions are tested simultaneously, their p-values are corrected using the Holm method.


In [12]:
def exact_mcnemar(df: pd.DataFrame,
                  column_a: str,
                  column_b: str,
                  label_a: str, 
                  label_b: str) -> dict:
    
    a = df[column_a].astype(bool).to_numpy()
    b = df[column_b].astype(bool).to_numpy()

    both_correct = int(np.sum(a & b))
    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))
    both_wrong = int(np.sum(~a & ~b))

    table = np.array([[both_correct, a_only],[b_only, both_wrong]])

    result = mcnemar(table, exact=True, correction=False)

    return {
        "comparison": f"{label_a} vs. {label_b}",
        "n": len(df),
        "accuracy_a": a.mean(),
        "accuracy_b": b.mean(),
        "difference_b_minus_a_pp": 100 * (b.mean() - a.mean()),
        "both_correct": both_correct,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "both_wrong": both_wrong,
        "p_raw": result.pvalue,
    }


In [13]:
secondary_results = [
    exact_mcnemar(
        analysis_df,
        column_a="central_correct",
        column_b="fea_correct",
        label_a="Central",
        label_b="FEA",
    ),
    exact_mcnemar(
        analysis_df,
        column_a="side_correct",
        column_b="fea_correct",
        label_a="Side",
        label_b="FEA",
    ),
    exact_mcnemar(
        analysis_df,
        column_a="central_correct",
        column_b="side_correct",
        label_a="Central",
        label_b="Side",
    ),
]

secondary_result_df = pd.DataFrame(secondary_results)

reject, p_holm, _, _ = multipletests(
    secondary_result_df["p_raw"],
    alpha=ALPHA,
    method="holm",
)

secondary_result_df["p_holm"] = p_holm
secondary_result_df["significant_holm"] = reject

secondary_result_df


,comparison,n,accuracy_a,accuracy_b,difference_b_minus_a_pp,both_correct,a_only_correct,b_only_correct,both_wrong,p_raw,p_holm,significant_holm
0,Central vs. FEA,378,0.727513,0.716931,-1.058201,212,63,59,44,0.786058,0.786058,False
1,Side vs. FEA,378,0.669312,0.716931,4.761905,202,51,69,56,0.120328,0.240656,False
2,Central vs. Side,378,0.727513,0.669312,-5.820106,217,58,36,67,0.029766,0.089299,False


## 8. Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled FEA-vs.-Image difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, pooled image accuracy, FEA accuracy, and the difference $\mathrm{Accuracy}_{FEA} - \mathrm{Accuracy}_{Image}$.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.


In [14]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        image_accuracy=("image_correct_mean", "mean"),
        fea_accuracy=("fea_correct", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["fea_accuracy"] - participant_result_df["image_accuracy"]
)

n_fea_better = int((participant_result_df["difference_pp"] > 0).sum())
n_image_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring FEA:   {n_fea_better}/8")
print(f"Participants favoring Image: {n_image_better}/8")
print(f"Participants tied:           {n_equal}/8")
print(f"Median participant difference (FEA - Image): {median_participant_difference_pp:.2f} percentage points")

participant_result_df


Participants favoring FEA:   4/8
Participants favoring Image: 3/8
Participants tied:           1/8
Median participant difference (FEA - Image): 3.70 percentage points


,participant_id,n_reenactments,image_accuracy,fea_accuracy,difference_pp
0,1,47,0.765957,0.574468,-19.148936
1,8,54,0.592593,0.666667,7.407407
2,10,46,0.489130,0.782609,29.347826
3,13,46,0.804348,0.804348,0.000000
4,15,48,0.687500,0.645833,-4.166667
5,18,43,0.802326,0.906977,10.465116
6,23,53,0.698113,0.830189,13.207547
7,27,41,0.780488,0.512195,-26.829268


## 9. Summary and Export

The primary result answers whether the reported pooled static image accuracy differs from the static FEA accuracy while respecting the reenactment-level dependency structure. The one-sample t-test provides a sensitivity check of the same mean difference. The secondary McNemar tests determine whether the result differs by camera perspective and whether central and side views themselves differ in classification accuracy. The participant-level breakdown is descriptive and is used to assess directional consistency and heterogeneity across the eight test participants.


In [15]:
summary_df = pd.DataFrame({
    "metric": [
        "Pooled image accuracy",
        "FEA accuracy",
        "Central accuracy",
        "Side accuracy",
        "FEA - pooled image difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring FEA",
        "Participants favoring Image",
        "Participants tied",
        "Median participant difference FEA - Image (pp)"
    ],
    "value": [
        pooled_image_accuracy,
        fea_accuracy,
        central_accuracy,
        side_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_fea_better,
        n_image_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df


,metric,value
0,Pooled image accuracy,0.698413
1,FEA accuracy,0.716931
2,Central accuracy,0.727513
3,Side accuracy,0.669312
4,FEA - pooled image difference (pp),1.851852
5,Primary bootstrap CI low (pp),-3.306878
6,Primary bootstrap CI high (pp),6.878307
7,Primary bootstrap p-value,0.493626
8,Sensitivity t statistic,0.708450
9,Sensitivity t-test p-value,0.479103


In [16]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_reenactment_table.csv", index=True)
primary_result_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_primary_result.csv", index=False)
sensitivity_result_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_sensitivity_ttest.csv", index=False)
secondary_result_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_secondary_mcnemar_results.csv", index=False)
participant_result_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_participant_level_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "static_image_vs_fea_summary.csv", index=False)

print(f"Results written to: {OUTPUT_DIR.resolve()}")


Results written to: /workspace/repos/emohevrdb-dfer/6_discussion/static-significance-tests/statistical_results
